# Capstone — Content Refresh Prioritization Research Paper

This notebook mirrors the deployed research paper and records the measured, public-safe analysis.


## 1. Question

**Research question:** Can observed content and search-performance signals be used to prioritize pages for human review for a possible content refresh?

**Decision supported:** The output supports human review and prioritization. It does not claim that a refresh will cause improved search performance.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../../data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', len(df.columns))


## 2. Data

The analysis uses the anonymized content-refresh dataset used during the track. The working file contains 30,000 page-level observations. Client IDs are used only for grouped validation. Outcome-derived fields are excluded from model features. No client names, private queries, or private URLs are disclosed.


In [ ]:
print('Unique clients:', df['client_id'].nunique())
print('Target distribution:')
print(df['trend_direction'].value_counts(dropna=False))


## 3. Methodology

**Target:** `trend_direction == down` is treated as the attention-positive class. This is a prioritization proxy, not a confirmed refresh-success label.

**Features:** search volume, competition level, CPC, word count, character count, 90-day impressions, clicks, pageviews, sessions, content age, and CTR. `trend_direction` and `trend_pct` are excluded because they contain outcome information.

**Model:** Logistic Regression was selected because it is simple, interpretable, and suitable for a first classification model.

**Validation:** Week-5 used an 80/20 client-grouped split with seed 42. The recorded evaluation contained 23,837 training rows and 6,163 test rows, with 25 training clients, 7 test clients, and zero client overlap.

**Baseline:** The Week-4 rule was compared using the same evaluation data and the same Average Precision and ROC-AUC metrics.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

target = 'trend_direction'
y = (df[target] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition_level', 'cpc', 'word_count',
    'char_count', 'impressions_90d', 'clicks_90d',
    'pageviews_90d', 'sessions_90d', 'content_age_days', 'ctr'
]
feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols].copy()
groups = df['client_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

model = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
p_model = model.predict_proba(X_test)[:, 1]

model_ap = average_precision_score(y_test, p_model)
model_auc = roc_auc_score(y_test, p_model)

print('Training rows:', len(train_idx))
print('Test rows:', len(test_idx))
print('Train clients:', groups.iloc[train_idx].nunique())
print('Test clients:', groups.iloc[test_idx].nunique())
print('Client overlap:', len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print('Measured Logistic Regression AP:', round(model_ap, 4))
print('Measured Logistic Regression ROC-AUC:', round(model_auc, 4))


## 4. Results (vs baseline)

The recorded Week-5 comparison measured Average Precision 0.5582 for Logistic Regression versus 0.4974 for the Week-4 baseline, and ROC-AUC 0.5621 versus 0.4644. These are observed validation results, not causal evidence that refreshing a page improves performance.


In [ ]:
baseline_ap = 0.4974
baseline_auc = 0.4644

results = pd.DataFrame({
    'method': ['W04 baseline rule', 'Logistic Regression'],
    'average_precision': [baseline_ap, model_ap],
    'roc_auc': [baseline_auc, model_auc]
})

print(results.to_string(index=False))
print('\nRecorded Week-5 values: AP=0.5582, ROC-AUC=0.5621')


## 5. Limitations

1. `trend_direction` is a proxy target, not confirmed refresh need or refresh success.
2. The analysis is observational and does not establish causality.
3. Search intent, competitors, content quality, and business priorities are not fully represented.
4. Results may differ for other clients or time periods.
5. The output is decision-support and must not automatically publish, rewrite, delete, redirect, or change production SEO settings.


## 6. Ranked recommendations

1. **Refresh first:** prioritize pages showing multiple observed refresh signals; a human reviewer verifies content quality and search intent.
2. **Monitor next:** review moderate-signal pages, such as pages with meaningful visibility but weaker CTR.
3. **Leave for now:** do not prioritize pages when available evidence is weak; revisit when signals change.

All recommendations are human-reviewed decision-support, not automatic actions.


In [ ]:
recommendations = pd.DataFrame({
    'rank': [1, 2, 3],
    'action': ['refresh_first', 'monitor_next', 'leave_for_now'],
    'reason_code': [
        'multiple observed refresh signals',
        'moderate visibility/CTR evidence',
        'limited evidence for immediate action'
    ],
    'human_review': [True, True, True]
})
print(recommendations.to_string(index=False))


## 7. Artifacts the paper embeds

The deployed paper uses aggregate model-vs-baseline metrics and public-safe charts. Individual pages, client identities, private queries, and private URLs are not included.


In [ ]:
paper_metrics = pd.DataFrame({
    'method': ['W04 baseline rule', 'Logistic Regression'],
    'average_precision': [baseline_ap, 0.5582],
    'roc_auc': [baseline_auc, 0.5621]
})

out = Path('../../work/outputs/capstone_metrics.csv')
out.parent.mkdir(parents=True, exist_ok=True)
paper_metrics.to_csv(out, index=False)

print('Saved:', out)
print(paper_metrics.to_string(index=False))


## Self-check

- [ ] Run all cells top-to-bottom with no errors.
- [ ] No client names, private queries, or private URLs appear.
- [ ] Claims use observed/measured/directional/decision-support language.
- [ ] The deployed paper contains all required sections and FlyRank data credit.
- [ ] `submission/paper_url.txt` contains the exact deployed URL on one line.
